# PokouAI — Merge LoRA checkpoint into base Gemma 4

Takes a saved LoRA checkpoint (from notebook 03's `save_steps=200` snapshots) and produces a merged HuggingFace model directory ready for `.litertlm` export via notebook 04.

**When to use this notebook**:

- Your training session timed out partway through (e.g., at step 4600 out of a planned 15,000) and you want to ship the partial fine-tune *now*, without continuing training.
- You need to reproduce a merge from a specific checkpoint without rerunning the full training pipeline.

**When NOT to use this notebook**:

- You want to *resume* training to completion. Use notebook 03 cell 14 with `CHECKPOINT_PATH` set and `SKIP_TRAINING = False` instead — that path preserves optimizer + scheduler + RNG state.

**Prereqs**: the checkpoint folder must contain `adapter_model.safetensors` (or `.bin`) and `adapter_config.json`. Either still in `/kaggle/working/` from a recent kernel, or attached as an Input dataset from a previous Version's output.

## 1 — Environment
Same install dance as notebook 03: force-reinstall unsloth + unsloth_zoo from git HEAD, upgrade torchao to ≥0.16.0, pin Pillow to 11.3.0 last so the unsloth deps don't undo it.

In [ ]:
!pip install -qU --force-reinstall --no-deps \
    "unsloth @ git+https://github.com/unslothai/unsloth.git" \
    "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
# Gemma 4 requires a recent transformers (per HF "Getting Started":
# `pip install -U transformers torch accelerate`). Older releases don't
# know the `gemma4` model_type, so AutoConfig returns None and unsloth
# surfaces this as the misleading
#     RuntimeError: Unsloth: No config file found
# error. Stay on latest. The regex hot-patch below covers the
# BitsAndBytesConfig.__init__ source format change that newer transformers
# introduced (which broke unsloth's monkey-patch).
!pip install -qU transformers accelerate peft trl bitsandbytes
!pip install -qU "torchao>=0.16.0"
# Belt-and-braces re-install of bitsandbytes — pip's resolver with --force-reinstall
# and --no-deps above can leave bitsandbytes uninstalled in some Kaggle environments.
# unsloth fails to import without it.
!pip install -qU bitsandbytes
!pip install --force-reinstall --no-cache-dir --no-deps "pillow==11.3.0"

# Hot-patch unsloth/models/_utils.py — required.
# Without this, `from unsloth import FastModel` raises:
#     AttributeError: 'NoneType' object has no attribute 'group'
# at unsloth/models/_utils.py:1797:
#     length_spaces = len(re.match(r"[\s]{1,}", BitsAndBytesConfig__init__[0]).group(0))
# That regex assumes BitsAndBytesConfig.__init__'s source starts with ≥1
# whitespace char (class-method indentation). Recent transformers ships
# its dispatched __init__ left-aligned, re.match returns None, .group(0)
# blows up. Swap {1,} → * so a zero-length prefix is valid; length_spaces
# becomes 0 and the dedent loop is a no-op (the right behavior).
# Use importlib.util.find_spec — it locates unsloth without importing it
# (importing it is what's broken).
import importlib.util, pathlib
_spec = importlib.util.find_spec('unsloth')
_utils = pathlib.Path(list(_spec.submodule_search_locations)[0]) / 'models' / '_utils.py'
_src = _utils.read_text()
_old = 're.match(r"[\\s]{1,}", BitsAndBytesConfig__init__[0]).group(0)'
_new = 're.match(r"[\\s]*",   BitsAndBytesConfig__init__[0]).group(0)'
if _old in _src:
    _utils.write_text(_src.replace(_old, _new))
    print(f'✓ patched {_utils} (BitsAndBytesConfig regex)')
elif _new in _src:
    print(f'✓ {_utils} already patched')
else:
    print(f'⚠ {_utils} regex line not found — unsloth shape changed; verify manually')

# Version checks query the *on-disk* package metadata, not what the kernel
# happens to have imported. Kaggle preloads some libs (e.g. torchao 0.10.0)
# at kernel startup, so `import torchao` returns the stale module even
# after pip has placed a new version on disk — and only a runtime restart
# clears that. Disk-level checks are restart-independent and tell us the
# install actually succeeded.
from importlib.metadata import version as _pkg_v
from packaging.version import Version

disk_pillow       = _pkg_v('pillow')
disk_torchao      = _pkg_v('torchao')
disk_transformers = _pkg_v('transformers')

assert disk_pillow == '11.3.0', f'on-disk Pillow: {disk_pillow}'
assert Version(disk_torchao) >= Version('0.16.0'), (
    f'on-disk torchao too old: {disk_torchao} — pip install above must have failed silently'
)

# bitsandbytes: import test stays as a real import — in some Kaggle
# environments pip reports it installed but the compiled C extension fails
# to load. That can only be detected by attempting the import.
try:
    import bitsandbytes as _bnb
    bnb_ok = True
except Exception as e:
    bnb_ok = False
    bnb_err = str(e)
assert bnb_ok, f'bitsandbytes failed to import: {bnb_err if not bnb_ok else "?"}'

print(f'✓ deps installed (on disk: Pillow {disk_pillow}, torchao {disk_torchao}, transformers {disk_transformers}, bitsandbytes ok)')
print('⚠ if first install in this session, RESTART RUNTIME before continuing — '
      'kernel may still have stale modules cached even though disk is correct')

## 2 — Parameters
Set `CHECKPOINT_PATH` to the directory containing `adapter_model.safetensors`. Common locations:

- `/kaggle/working/cocoa_v1_e2b/checkpoint-4600` — current kernel's training output
- `/kaggle/input/<your-dataset-slug>/cocoa_v1_e2b/checkpoint-4600` — attached from a previous Version

If unsure, run `!find /kaggle/input /kaggle/working -name "adapter_model.safetensors" 2>/dev/null` in a fresh cell.

In [ ]:
VARIANT = 'e2b'   # must match what was used in notebook 03 ('e2b' or 'e4b')
MAX_SEQ_LEN = 2048

# ↓↓↓ EDIT THIS to point at your checkpoint folder ↓↓↓
CHECKPOINT_PATH = '/kaggle/input/notebooks/yaokouadio/pokou-ai-cocoa-finetune/cocoa_v1_e2b/checkpoint-4600'

OUTPUT_DIR = f'/kaggle/working/cocoa_v1_{VARIANT}_merged'

import os, json, time, torch, subprocess, gc
from pathlib import Path

assert os.path.isdir(CHECKPOINT_PATH), f'missing: {CHECKPOINT_PATH}'
adapter_safe = os.path.join(CHECKPOINT_PATH, 'adapter_model.safetensors')
adapter_bin  = os.path.join(CHECKPOINT_PATH, 'adapter_model.bin')
if not os.path.exists(adapter_safe) and not os.path.exists(adapter_bin):
    raise FileNotFoundError(f'no adapter_model.{{safetensors,bin}} in {CHECKPOINT_PATH}')

# Quick peek at training history embedded in the checkpoint — lets you sanity-check the run
ts_path = os.path.join(CHECKPOINT_PATH, 'trainer_state.json')
if os.path.exists(ts_path):
    s = json.load(open(ts_path))
    losses = [l['loss'] for l in s.get('log_history', []) if 'loss' in l]
    print(f'checkpoint:  step {s.get("global_step")}')
    if losses:
        print(f'first loss:  {losses[0]:.3f}')
        print(f'last loss:   {losses[-1]:.3f}')
        print(f'last 5:      {[round(l,3) for l in losses[-5:]]}')
        if losses[-1] > losses[0] * 0.85:
            print('⚠ loss barely moved — model may be undercooked. Merge will still work but quality will be limited.')

# Hard disk-free gate. The merged 16-bit dir is ~5–6 GB for E2B (~9–10 GB
# for E4B) plus a write buffer; previous runs truncated mid-save when
# /kaggle/working ran out of room and produced unreadable shards. Fail
# fast here instead of 5 minutes into the merge.
import shutil
free_gb = shutil.disk_usage('/kaggle/working').free / 1e9
print(f'\nfree disk: {free_gb:.1f} GB')
assert free_gb >= 12, (
    f'only {free_gb:.1f} GB free in /kaggle/working — merge save needs ≥ 12 GB '
    f'(merged weights ~6 GB + safetensors write buffer + safety margin). '
    f'Free up space first:\n'
    f'  !rm -rf /kaggle/working/cocoa_v1_e2b_merged          # last failed merge\n'
    f'  !rm -rf /kaggle/working/cocoa_v1_e2b-mmproj.gguf     # if present\n'
    f'  !rm -rf /kaggle/working/llama.cpp /kaggle/working/state.db'
)

In [ ]:
# === HF auth (gated Gemma 4) ===
#
# ⚠ SECURITY: pasting a token below writes it into the .ipynb file on disk.
#    Anything that opens the .ipynb (git commits, Kaggle Versions, file
#    sharing, transcript dumps) will see this value. Before committing or
#    sharing this notebook, replace the value with '' again — OR migrate to
#    Kaggle Secrets (`kaggle_secrets.UserSecretsClient().get_secret(...)`).
#
# Token only needs *Read* scope. Mint at:
#     https://huggingface.co/settings/tokens
# License acknowledgement is SEPARATE from the token — visit the model
# page (e.g. huggingface.co/google/gemma-4-E2B-it) and click
# "Acknowledge license" once per HF account, or downloads will still 403.
import os

HF_TOKEN = ''  # ← paste your hf_xxx token here

assert HF_TOKEN.startswith('hf_'), (
    'HF_TOKEN is empty/malformed. Paste your token (starts with hf_) above, '
    'or set it in Kaggle Secrets and load via UserSecretsClient.'
)
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN  # huggingface_hub also reads this name
print(f'✓ HF_TOKEN set ({len(HF_TOKEN)} chars, ends in …{HF_TOKEN[-4:]})')

## 3 — Load base + apply LoRA structure
Same flow as notebook 03 §2 + §2.5. We re-create the LoRA structure with the same hyperparameters that were used for training (`r=16, lora_alpha=32, ...`) so the saved adapter weights load cleanly into the matching shape. **These params must match notebook 03 cell 7 exactly** — if you tuned them there, mirror the change here.

In [ ]:
import os
from unsloth import FastModel

# HF repo IDs use UPPERCASE variant suffix (E2B / E4B).
# Direct: google/gemma-4-E2B-it (gated — needs HF_TOKEN + license accept)
# Fallback (pre-quantized 4-bit): unsloth/gemma-4-E2B-it-unsloth-bnb-4bit
DIRECT_REPO   = f'google/gemma-4-{VARIANT.upper()}-it'
FALLBACK_REPO = f'unsloth/gemma-4-{VARIANT.upper()}-it-unsloth-bnb-4bit'

# Pass token explicitly — unsloth's internal config fetcher does not
# always read HF_TOKEN from the env, especially when the model is gated
# (gemma-4 is). When the fetch silently 401s, unsloth surfaces it as the
# generic "Unsloth: No config file found" RuntimeError, which is
# indistinguishable from a 404 in the trace.
HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
assert HF_TOKEN, 'HF_TOKEN not set — run the auth cell above first'

def load_model():
    try:
        print(f'trying direct load: {DIRECT_REPO}')
        return FastModel.from_pretrained(
            model_name=DIRECT_REPO,
            max_seq_length=MAX_SEQ_LEN,
            load_in_4bit=True,
            dtype=None,
            token=HF_TOKEN,
        )
    except Exception as e:
        print(f'direct load failed ({type(e).__name__}: {e}) — falling back to {FALLBACK_REPO}')
        return FastModel.from_pretrained(
            model_name=FALLBACK_REPO,
            max_seq_length=MAX_SEQ_LEN,
            load_in_4bit=True,
            dtype=None,
            token=HF_TOKEN,
        )

model, tokenizer = load_model()
print(f'✓ loaded base {VARIANT.upper()}')

model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    finetune_vision_layers=True,
    finetune_language_layers=True,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

## 4 — Load adapter weights from checkpoint
Overwrites the random LoRA initialization with the trained weights. `strict=False` is intentional: the saved file only contains LoRA tensors, not the (frozen) base weights, so the base parameters end up in the "unexpected keys" list — that's fine.

In [ ]:
if os.path.exists(adapter_safe):
    from safetensors.torch import load_file
    adapter_state = load_file(adapter_safe)
    print(f'loaded {len(adapter_state)} tensors from adapter_model.safetensors')
else:
    adapter_state = torch.load(adapter_bin, map_location='cuda')
    print(f'loaded {len(adapter_state)} tensors from adapter_model.bin')

missing, unexpected = model.load_state_dict(adapter_state, strict=False)
print(f'  missing keys (base params, expected):    {len(missing)}')
print(f'  unexpected keys (should be ~0):          {len(unexpected)}')
if len(unexpected) > 0:
    print(f'  first 3 unexpected: {unexpected[:3]}')
    print('  ⚠ unexpected keys mean the LoRA structure here doesn\'t match the saved checkpoint.')
    print('     Most likely cause: target_modules, r, or lora_alpha differs from the training run.')
print('\n✓ adapter loaded into model')

## 5 — Merge LoRA into base + save HF
Mirrors notebook 03 cell 18: empties CUDA cache first, then `save_pretrained_merged` writes the merged 16-bit HuggingFace dir to `OUTPUT_DIR`. Expected output size: ~5–6 GB for E2B, ~9–10 GB for E4B.

In [ ]:
import shutil

# Nuke any partial merge from a previous run. Partial shards from a
# truncated save are unrecoverable — they look like valid safetensors
# files but the header claims more bytes than the file actually contains,
# and `litert-torch export_hf` then fails with "incomplete metadata".
if os.path.exists(OUTPUT_DIR):
    print(f'removing stale {OUTPUT_DIR}')
    shutil.rmtree(OUTPUT_DIR)

gc.collect()
torch.cuda.empty_cache()
print(f'GPU mem before merge: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
print(f'free disk: {shutil.disk_usage("/kaggle/working").free / 1e9:.1f} GB')

print(f'\nsaving merged model to {OUTPUT_DIR}...')
t0 = time.time()
# max_shard_size="1GB" → ~6 small shards instead of 5 big ones for E2B.
# Smaller shards = each write is shorter = less window where an OOM /
# timeout / disk-pressure event leaves a truncated header. The previous
# run corrupted a 2 GB shard mid-write; this should prevent recurrence.
try:
    model.save_pretrained_merged(
        OUTPUT_DIR, tokenizer,
        save_method='merged_16bit',
        max_shard_size='1GB',
    )
except TypeError:
    # Older unsloth versions don't accept max_shard_size — fall back to
    # default sharding. Verification block below will still catch corruption.
    print('  (max_shard_size not supported on this unsloth — using defaults)')
    model.save_pretrained_merged(OUTPUT_DIR, tokenizer, save_method='merged_16bit')
elapsed = time.time() - t0

size = subprocess.check_output(['du', '-sh', OUTPUT_DIR]).decode().split()[0]
print(f'\n✓ merged model saved in {elapsed/60:.1f} min ({size})')
!ls -lh {OUTPUT_DIR}/ | head -20

# Verify every shard's header before moving on. Catches the
# "incomplete metadata, file not fully covered" failure at its source —
# much cheaper than discovering it 15 min into notebook 04.
from safetensors import safe_open
import glob
shards = sorted(glob.glob(f'{OUTPUT_DIR}/*.safetensors'))
print(f'\nverifying {len(shards)} shard(s)...')
for path in shards:
    sz_mb = os.path.getsize(path) / 1e6
    with safe_open(path, framework='pt') as f:
        n_keys = len(list(f.keys()))
    print(f'  ✓ {sz_mb:7.1f} MB  {n_keys:4d} keys  {os.path.basename(path)}')
print(f'\n✓ all shards verified — safe to run notebook 04')

## 6 — Sanity check (text-only)
Quick generation from a hardcoded French prompt — confirms the merged model produces coherent, structured output before you commit to the `.litertlm` export in notebook 04. Multimodal inference happens in notebook 04's smoke-test step (via `litert-lm run`) and on-device via `react-native-litert-lm`'s `sendMessageWithImage`.

In [ ]:
from transformers import TextStreamer

messages = [{
    'role': 'user',
    'content': [
        {'type': 'text', 'text': (
            'Tu es un agronome cacao. Quels sont les symptômes typiques de la pourriture brune '
            'des cabosses (Phytophthora) ? Réponds au format MALADIE / SYMPTOMES / TRAITEMENT / PREVENTION.'
        )},
    ],
}]
# return_dict=True → returns {'input_ids', 'attention_mask'}, silencing
# the transformers warning when pad_token == eos_token (true for Gemma).
# Without an explicit attention_mask, generate() can produce subtly wrong
# logits on right-padded batches; harmless for this single greedy sample
# but worth getting right.
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors='pt',
    tokenize=True,
    return_dict=True,
).to('cuda')

# This is a TEXT-ONLY smoke test. The fine-tune was trained on image+text
# rows, so on text-only prompts the model falls back to base-Gemma's
# verbose, hierarchical style and expands the 4 sections with sub-bullets
# (~1200–1500 tokens for this prompt). The strict MALADIE/SYMPTOMES/
# TRAITEMENT/PREVENTION format will reassert itself on actual image
# queries (tested in notebook 04 via `litert-lm run`, and on-device via
# `react-native-litert-lm`'s sendMessageWithImage). Cap at 1500 so the
# smoke test completes; don't read format-compliance into the output.
streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(**inputs, max_new_tokens=1500, streamer=streamer, do_sample=False)

## 7 — Next: export to `.litertlm`

Open `04_quantize_litert_lm.ipynb` in this same Kaggle session — it'll find the merged dir at `/kaggle/working/cocoa_v1_e2b_merged/` and produce `model.litertlm` (~2.5 GB E2B int4, multimodal included).

Save Version (Commit) before the kernel times out so the bundle persists into the version output. Download from there → sideload to the phone, or push to a private HF repo for first-launch auto-download.